# Eval: medir resultados

Inputs y expectativas → ejecución y comparación exacta → informe de casos.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3", "Instala el wheel local 2.1.3: no hay fallback a 2.1.2"


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [3]:
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
cases = ({"input": {"value": 3}, "expected": {"ok": True, "data": {"value": 8}}},
         {"input": {"value": -3}, "expected": {"ok": True, "data": {"value": -4}}})
result = toolkit.Eval(name="contract", unit=agent, cases=cases).run()
assert result.ok and result.data["passed"] == 2
wrong = toolkit.Eval("negative", agent, ({"input": {"value": 3}, "expected": {"data": {"value": 999}}},)).run()
assert not wrong.ok and wrong.data["failed"] == 1 and wrong.children[0].ok
print({"passed": result.data["passed"], "wrong_expectation_detected": not wrong.ok})

{'passed': 2, 'wrong_expectation_detected': True}


Eval comprueba `expected.ok` y `expected.data` exactos y conserva las ejecuciones evaluadas. Rechaza claves de expectativas desconocidas. Es validación determinista, sin juez LLM. Eval recibe un Agent/System directamente; no evalúa el wrapper Environment.

In [4]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
